# Worked Example: Time-Frequency Reward Contrast

## Goal
Morlet TFR on feedback epochs via `build_analysis_config` / `run_analysis`, with reward vs no-reward difference map.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from LFPAnalysis import build_analysis_config, load_lfp, run_analysis
from LFPAnalysis.config import LoadConfig

beh = pd.read_csv(Path('../../data/sample_beh.csv'))
epochs = load_lfp(
    LoadConfig(path=Path('../../data/sample_feedback_start-epo.fif'), file_format='mne', preload=True)
)
epochs.metadata = beh[['reward', 'rpe']]
chan = 'racas1-racas2'
freqs = np.arange(4, 30, 4).tolist()
tfr_cfg = build_analysis_config(tfr_method='morlet', tfr_freqs=freqs, tfr_n_cycles=3.0)
reward_result = run_analysis(epochs['reward == 1'].copy().pick([chan]), tfr_cfg)
loss_result = run_analysis(epochs['reward == 0'].copy().pick([chan]), tfr_cfg)
reward_power = reward_result.tfr['power'].average()
loss_power = loss_result.tfr['power'].average()
diff = reward_power.data[0] - loss_power.data[0]
print('TFR shape:', reward_power.data.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
freq_arr = np.asarray(freqs)
for ax, data, title in zip(
    axes,
    [reward_power.data[0], loss_power.data[0], diff],
    ['reward', 'no reward', 'difference'],
):
    im = ax.imshow(
        data,
        aspect='auto',
        origin='lower',
        extent=[reward_power.times[0], reward_power.times[-1], freq_arr[0], freq_arr[-1]],
        cmap='RdBu_r',
    )
    ax.axvline(0, color='k', ls='--', lw=0.8)
    ax.set(xlabel='Time (s)', title=title)
axes[0].set_ylabel('Frequency (Hz)')
fig.colorbar(im, ax=axes, shrink=0.8, label='Power')
fig.tight_layout()
plt.show()

## Saving results

See chapter 15 (`15_saving_and_organizing_results`) for the recommended `results/` layout.

In [ ]:
# Uncomment to save. See chapter 15 for the recommended results/ layout.
# out = Path('../../results/worked-examples')
# out.mkdir(parents=True, exist_ok=True)
# reward_power.save(out / 'reward-tfr.h5', overwrite=True)
# loss_power.save(out / 'no_reward-tfr.h5', overwrite=True)
# Reload with: mne.time_frequency.read_tfrs(out / 'reward-tfr.h5')

## Next step

Chapter 10 (`10_first_connectivity_and_surrogates`) covers connectivity.